In [1]:
!pip -q install xgboost shap scikit-learn --upgrade rasterio fiona
!apt-get install -y libgdal-dev
!pip install GDAL==3.4.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 61.9 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgdal-dev is already the newest version (3.8.4+dfsg-1~jammy0).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.6/757.6 kB 35.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for GDAL
  Running setup.py clean for GDAL
Failed to build GDAL
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects 

# Split the SOC samples (Dataset-II)

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
from sklearn.model_selection import train_test_split

# =====================================================
# USER SETTINGS
# =====================================================
input_file = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/SOC_GEE_exports/"
    "LUCAS_samples_GSE.geojson"
)

output_folder = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/1_Feature_Selection/"
    "Split_features"
)

# Percentage reserved for completely independent validation
validation_size = 0.10

# Reproducible random split
random_seed = 42

# Required output coordinate reference system
output_crs = "EPSG:3763"

# Only use this when the input file has no CRS metadata and you are
# certain that its coordinates are longitude/latitude in WGS 84.
source_crs_if_missing = "EPSG:4326"

# =====================================================
# READ INPUT SPATIAL DATA
# =====================================================
if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"Input file was not found:\n{input_file}"
    )

gdf = gpd.read_file(input_file)

if gdf.empty:
    raise ValueError("The input spatial dataset contains no features.")

print("=" * 65)
print("INPUT DATA")
print("=" * 65)
print(f"Input file:       {input_file}")
print(f"Number of rows:   {len(gdf):,}")
print(f"Number of fields: {len(gdf.columns):,}")
print(f"Original CRS:     {gdf.crs}")
print(
    "Geometry types:  "
    f"{gdf.geometry.geom_type.dropna().unique().tolist()}"
)

# =====================================================
# CHECK AND DEFINE SOURCE CRS
# =====================================================
if gdf.crs is None:
    if source_crs_if_missing is None:
        raise ValueError(
            "The input dataset has no CRS information. Set "
            "'source_crs_if_missing' to the dataset's actual source CRS."
        )

    print(
        f"\nWarning: the input has no CRS metadata. Assigning "
        f"{source_crs_if_missing} as the source CRS."
    )

    # This assigns the known source CRS without transforming coordinates
    gdf = gdf.set_crs(source_crs_if_missing)

# =====================================================
# GEOMETRY QUALITY CHECKS
# =====================================================
missing_geometry_count = int(gdf.geometry.isna().sum())
empty_geometry_count = int(gdf.geometry.is_empty.sum())

if missing_geometry_count > 0:
    raise ValueError(
        f"The input contains {missing_geometry_count} rows "
        "with missing geometry."
    )

if empty_geometry_count > 0:
    raise ValueError(
        f"The input contains {empty_geometry_count} rows "
        "with empty geometry."
    )

# This workflow is intended for point samples
non_point_mask = ~gdf.geometry.geom_type.isin(
    ["Point", "MultiPoint"]
)

if non_point_mask.any():
    non_point_types = (
        gdf.loc[non_point_mask, gdf.geometry.name]
        .geom_type
        .unique()
        .tolist()
    )

    raise ValueError(
        "The dataset contains non-point geometries: "
        f"{non_point_types}"
    )

# =====================================================
# REPROJECT TO EPSG:3763
# =====================================================
if gdf.crs.to_string() != output_crs:
    gdf = gdf.to_crs(output_crs)

if gdf.crs.to_epsg() != 3763:
    raise RuntimeError(
        f"Reprojection failed. Current CRS is {gdf.crs}, "
        "but EPSG:3763 was required."
    )

print(f"Reprojected CRS:  {gdf.crs}")

# =====================================================
# CREATE TEMPORARY UNIQUE ROW IDENTIFIER
# =====================================================
# This ID is used only to verify that no feature is present
# in both the development and validation datasets.
temporary_id = "_split_row_id"

while temporary_id in gdf.columns:
    temporary_id = "_" + temporary_id

gdf = gdf.copy()
gdf[temporary_id] = np.arange(len(gdf), dtype=np.int64)

# =====================================================
# RANDOM 90% / 10% SPLIT
# =====================================================
all_row_indices = np.arange(len(gdf))

development_indices, validation_indices = train_test_split(
    all_row_indices,
    test_size=validation_size,
    random_state=random_seed,
    shuffle=True
)

gdf_development = gdf.iloc[development_indices].copy()
gdf_validation = gdf.iloc[validation_indices].copy()

# Randomize row order within each output dataset
gdf_development = (
    gdf_development
    .sample(frac=1, random_state=random_seed)
    .reset_index(drop=True)
)

gdf_validation = (
    gdf_validation
    .sample(frac=1, random_state=random_seed)
    .reset_index(drop=True)
)

# =====================================================
# DATA-LEAKAGE AND COMPLETENESS CHECKS
# =====================================================
development_ids = set(gdf_development[temporary_id])
validation_ids = set(gdf_validation[temporary_id])

overlapping_ids = development_ids.intersection(validation_ids)
combined_ids = development_ids.union(validation_ids)

if overlapping_ids:
    raise RuntimeError(
        f"Split error: {len(overlapping_ids)} input rows occur "
        "in both output datasets."
    )

if len(combined_ids) != len(gdf):
    raise RuntimeError(
        "Split error: some input rows are missing from the outputs."
    )

if len(gdf_development) + len(gdf_validation) != len(gdf):
    raise RuntimeError(
        "Split error: output row totals do not equal the input row total."
    )

# Remove the temporary identifier before export
gdf_development = gdf_development.drop(columns=[temporary_id])
gdf_validation = gdf_validation.drop(columns=[temporary_id])

# Explicitly retain geometry and EPSG:3763
gdf_development = gpd.GeoDataFrame(
    gdf_development,
    geometry=gdf.geometry.name,
    crs=output_crs
)

gdf_validation = gpd.GeoDataFrame(
    gdf_validation,
    geometry=gdf.geometry.name,
    crs=output_crs
)

# =====================================================
# PREPARE OUTPUT PATHS
# =====================================================
Path(output_folder).mkdir(parents=True, exist_ok=True)

base_name = Path(input_file).stem

development_path = os.path.join(
    output_folder,
    f"{base_name}_train_test.gpkg"
)

validation_path = os.path.join(
    output_folder,
    f"{base_name}_validation.gpkg"
)

# Remove previous output files before rewriting
for output_path in [development_path, validation_path]:
    if os.path.exists(output_path):
        os.remove(output_path)

# =====================================================
# EXPORT GEOPACKAGE FILES
# =====================================================
gdf_development.to_file(
    development_path,
    layer="development_90_percent",
    driver="GPKG",
    index=False
)

gdf_validation.to_file(
    validation_path,
    layer="independent_validation_10_percent",
    driver="GPKG",
    index=False
)

# =====================================================
# READ OUTPUTS BACK FOR FINAL VERIFICATION
# =====================================================
check_development = gpd.read_file(
    development_path,
    layer="development_90_percent"
)

check_validation = gpd.read_file(
    validation_path,
    layer="independent_validation_10_percent"
)

if check_development.crs is None or check_development.crs.to_epsg() != 3763:
    raise RuntimeError(
        "The development GeoPackage was not exported in EPSG:3763."
    )

if check_validation.crs is None or check_validation.crs.to_epsg() != 3763:
    raise RuntimeError(
        "The validation GeoPackage was not exported in EPSG:3763."
    )

if len(check_development) != len(gdf_development):
    raise RuntimeError(
        "The exported development row count is incorrect."
    )

if len(check_validation) != len(gdf_validation):
    raise RuntimeError(
        "The exported validation row count is incorrect."
    )

# =====================================================
# FINAL REPORT
# =====================================================
development_percentage = (
    len(gdf_development) / len(gdf) * 100
)

validation_percentage = (
    len(gdf_validation) / len(gdf) * 100
)

print("\n" + "=" * 65)
print("RANDOM SPLIT COMPLETED SUCCESSFULLY")
print("=" * 65)
print(f"Random seed:                    {random_seed}")
print(f"Output CRS:                     {gdf.crs}")
print()
print(f"Original number of points:      {len(gdf):,}")
print(
    f"Development points:             "
    f"{len(gdf_development):,} "
    f"({development_percentage:.2f}%)"
)
print(
    f"Independent validation points:  "
    f"{len(gdf_validation):,} "
    f"({validation_percentage:.2f}%)"
)
print(f"Overlapping input rows:         {len(overlapping_ids)}")
print()
print("Development GeoPackage:")
print(development_path)
print()
print("Independent validation GeoPackage:")
print(validation_path)

INPUT DATA
Input file:       /content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/LUCAS_samples_GSE.geojson
Number of rows:   428
Number of fields: 93
Original CRS:     EPSG:4326
Geometry types:  ['Point']
Reprojected CRS:  EPSG:3763

RANDOM SPLIT COMPLETED SUCCESSFULLY
Random seed:                    42
Output CRS:                     EPSG:3763

Original number of points:      428
Development points:             385 (89.95%)
Independent validation points:  43 (10.05%)
Overlapping input rows:         0

Development GeoPackage:
/content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/1_Feature_Selection/Split_features/LUCAS_samples_GSE_train_test.gpkg

Independent validation GeoPackage:
/content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/1_Feature_Selection/Split_features/LUCAS_samples_GSE_validation.gpkg


Improve the attribute table

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

import fiona
import geopandas as gpd

# =====================================================
# USER SETTINGS
# =====================================================
input_folder = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/1_Feature_Selection/"
    "Split_features"
)

# Attributes to retain in addition to the embedding bands
required_columns = [
    "OC",
    "LC0_Desc",
    "LC1_Desc",
    "LU1_Desc"
]

# Keep columns such as A01, A02, A03, etc.
embedding_prefix = "A"

# Required output CRS
target_crs = "EPSG:3763"

# Create backup copies before replacing the original files
create_backup = True

# =====================================================
# CHECK INPUT FOLDER
# =====================================================
input_folder_path = Path(input_folder)

if not input_folder_path.exists():
    raise FileNotFoundError(
        f"Input folder was not found:\n{input_folder}"
    )

# Find all GeoPackage files, excluding previously created backups
gpkg_files = sorted([
    path for path in input_folder_path.glob("*.gpkg")
    if not path.stem.endswith("_backup")
])

if not gpkg_files:
    raise FileNotFoundError(
        f"No GeoPackage files were found in:\n{input_folder}"
    )

print("=" * 70)
print("GEOPACKAGE FILES FOUND")
print("=" * 70)

for gpkg_file in gpkg_files:
    print(f"  - {gpkg_file.name}")

print(f"\nTotal files: {len(gpkg_files)}")

# =====================================================
# PROCESS FUNCTION
# =====================================================
def clean_geopackage(gpkg_path):
    """
    Retain:
      - OC
      - LC0_Desc
      - LC1_Desc
      - LU1_Desc
      - all columns starting with A
      - geometry

    Remove all other attributes and replace the original GeoPackage.
    """

    gpkg_path = Path(gpkg_path)

    print("\n" + "=" * 70)
    print(f"PROCESSING: {gpkg_path.name}")
    print("=" * 70)

    # -------------------------------------------------
    # IDENTIFY LAYERS
    # -------------------------------------------------
    layers = fiona.listlayers(str(gpkg_path))

    if len(layers) == 0:
        raise ValueError(
            f"No layers were found in:\n{gpkg_path}"
        )

    if len(layers) > 1:
        raise ValueError(
            f"{gpkg_path.name} contains multiple layers: {layers}\n"
            "This code expects one spatial layer per GeoPackage."
        )

    layer_name = layers[0]

    # -------------------------------------------------
    # READ DATA
    # -------------------------------------------------
    gdf = gpd.read_file(
        gpkg_path,
        layer=layer_name
    )

    if gdf.empty:
        raise ValueError(
            f"{gpkg_path.name} contains no features."
        )

    geometry_column = gdf.geometry.name

    print(f"Layer:             {layer_name}")
    print(f"Original features: {len(gdf):,}")
    print(f"Original columns:  {len(gdf.columns):,}")
    print(f"Original CRS:      {gdf.crs}")

    # -------------------------------------------------
    # CRS CHECK
    # -------------------------------------------------
    if gdf.crs is None:
        raise ValueError(
            f"{gpkg_path.name} does not have a defined CRS."
        )

    if gdf.crs.to_epsg() != 3763:
        print(
            f"Reprojecting {gpkg_path.name} from "
            f"{gdf.crs} to {target_crs}..."
        )

        gdf = gdf.to_crs(target_crs)

    if gdf.crs.to_epsg() != 3763:
        raise RuntimeError(
            f"Could not reproject {gpkg_path.name} to EPSG:3763."
        )

    # -------------------------------------------------
    # GEOMETRY CHECKS
    # -------------------------------------------------
    missing_geometries = int(gdf.geometry.isna().sum())
    empty_geometries = int(gdf.geometry.is_empty.sum())

    if missing_geometries > 0:
        raise ValueError(
            f"{gpkg_path.name} contains "
            f"{missing_geometries} missing geometries."
        )

    if empty_geometries > 0:
        raise ValueError(
            f"{gpkg_path.name} contains "
            f"{empty_geometries} empty geometries."
        )

    # -------------------------------------------------
    # FIND REQUESTED COLUMNS
    # -------------------------------------------------
    # Case-insensitive lookup while retaining original names
    column_lookup = {
        str(column).lower(): column
        for column in gdf.columns
    }

    retained_required_columns = []
    missing_required_columns = []

    for requested_column in required_columns:
        actual_column = column_lookup.get(
            requested_column.lower()
        )

        if actual_column is not None:
            retained_required_columns.append(actual_column)
        else:
            missing_required_columns.append(requested_column)

    if missing_required_columns:
        print("\nWarning—requested columns not found:")

        for column in missing_required_columns:
            print(f"  - {column}")

    # Keep all A-prefixed embedding attributes
    embedding_columns = [
        column
        for column in gdf.columns
        if column != geometry_column
        and str(column).upper().startswith(
            embedding_prefix.upper()
        )
    ]

    # Final column order:
    # target/descriptive fields, embeddings, geometry
    columns_to_keep = (
        retained_required_columns
        + embedding_columns
        + [geometry_column]
    )

    # Remove duplicate entries while preserving order
    columns_to_keep = list(
        dict.fromkeys(columns_to_keep)
    )

    removed_columns = [
        column
        for column in gdf.columns
        if column not in columns_to_keep
    ]

    # -------------------------------------------------
    # CREATE CLEANED GEODATAFRAME
    # -------------------------------------------------
    cleaned_gdf = gdf[columns_to_keep].copy()

    cleaned_gdf = gpd.GeoDataFrame(
        cleaned_gdf,
        geometry=geometry_column,
        crs=target_crs
    )

    # Confirm that an id column was not retained
    remaining_id_columns = [
        column
        for column in cleaned_gdf.columns
        if str(column).lower() == "id"
    ]

    if remaining_id_columns:
        raise RuntimeError(
            f"The id attribute was not removed from "
            f"{gpkg_path.name}."
        )

    print(f"\nEmbedding fields retained: {len(embedding_columns)}")
    print(f"Total fields retained:     {len(cleaned_gdf.columns)}")
    print(f"Total fields removed:      {len(removed_columns)}")

    print("\nRetained attributes:")
    for column in cleaned_gdf.columns:
        print(f"  + {column}")

    print("\nRemoved attributes:")
    for column in removed_columns:
        print(f"  - {column}")

    # -------------------------------------------------
    # CREATE BACKUP
    # -------------------------------------------------
    if create_backup:
        backup_path = gpkg_path.with_name(
            f"{gpkg_path.stem}_backup.gpkg"
        )

        # Create the backup only if one does not already exist
        if not backup_path.exists():
            shutil.copy2(
                gpkg_path,
                backup_path
            )

            print(f"\nBackup created:\n{backup_path}")
        else:
            print(
                f"\nBackup already exists and was not overwritten:\n"
                f"{backup_path}"
            )

    # -------------------------------------------------
    # WRITE TO TEMPORARY FILE
    # -------------------------------------------------
    temporary_directory = tempfile.mkdtemp(
        dir=str(gpkg_path.parent)
    )

    temporary_gpkg = Path(
        temporary_directory
    ) / gpkg_path.name

    cleaned_gdf.to_file(
        temporary_gpkg,
        layer=layer_name,
        driver="GPKG",
        index=False
    )

    # -------------------------------------------------
    # VERIFY TEMPORARY FILE
    # -------------------------------------------------
    check_gdf = gpd.read_file(
        temporary_gpkg,
        layer=layer_name
    )

    if len(check_gdf) != len(gdf):
        raise RuntimeError(
            f"Feature count changed for {gpkg_path.name}."
        )

    if check_gdf.crs is None or check_gdf.crs.to_epsg() != 3763:
        raise RuntimeError(
            f"The temporary output for {gpkg_path.name} "
            "is not in EPSG:3763."
        )

    if list(check_gdf.columns) != list(cleaned_gdf.columns):
        raise RuntimeError(
            f"Exported columns do not match for "
            f"{gpkg_path.name}."
        )

    # -------------------------------------------------
    # REPLACE ORIGINAL FILE
    # -------------------------------------------------
    os.replace(
        temporary_gpkg,
        gpkg_path
    )

    shutil.rmtree(
        temporary_directory,
        ignore_errors=True
    )

    # -------------------------------------------------
    # FINAL VERIFICATION
    # -------------------------------------------------
    final_gdf = gpd.read_file(
        gpkg_path,
        layer=layer_name
    )

    print("\nFile updated successfully.")
    print(f"Updated file:       {gpkg_path}")
    print(f"Features retained:  {len(final_gdf):,}")
    print(f"Fields retained:    {len(final_gdf.columns):,}")
    print(f"Final CRS:          {final_gdf.crs}")

    return {
        "file": gpkg_path.name,
        "features": len(final_gdf),
        "embedding_columns": len(embedding_columns),
        "retained_columns": len(final_gdf.columns),
        "removed_columns": len(removed_columns),
        "crs": str(final_gdf.crs)
    }


# =====================================================
# PROCESS ALL GPKG FILES
# =====================================================
processing_results = []

for gpkg_file in gpkg_files:
    try:
        result = clean_geopackage(gpkg_file)
        processing_results.append(result)

    except Exception as error:
        print("\n" + "!" * 70)
        print(f"ERROR PROCESSING: {gpkg_file.name}")
        print(error)
        print("!" * 70)

# =====================================================
# FINAL SUMMARY
# =====================================================
print("\n" + "=" * 70)
print("BATCH PROCESSING SUMMARY")
print("=" * 70)

if processing_results:
    for result in processing_results:
        print(f"\nFile: {result['file']}")
        print(f"  Features:          {result['features']:,}")
        print(f"  Embedding fields:  {result['embedding_columns']}")
        print(f"  Fields retained:   {result['retained_columns']}")
        print(f"  Fields removed:    {result['removed_columns']}")
        print(f"  CRS:               {result['crs']}")

    print(
        f"\nSuccessfully processed "
        f"{len(processing_results)} of {len(gpkg_files)} files."
    )
else:
    print("No files were processed successfully.")

GEOPACKAGE FILES FOUND
  - LUCAS_samples_GSE_train_test.gpkg
  - LUCAS_samples_GSE_validation.gpkg

Total files: 2

PROCESSING: LUCAS_samples_GSE_train_test.gpkg
Layer:             development_90_percent
Original features: 385
Original columns:  93
Original CRS:      EPSG:3763

Embedding fields retained: 64
Total fields retained:     69
Total fields removed:      24

Retained attributes:
  + OC
  + LC0_Desc
  + LC1_Desc
  + LU1_Desc
  + A00
  + A01
  + A02
  + A03
  + A04
  + A05
  + A06
  + A07
  + A08
  + A09
  + A10
  + A11
  + A12
  + A13
  + A14
  + A15
  + A16
  + A17
  + A18
  + A19
  + A20
  + A21
  + A22
  + A23
  + A24
  + A25
  + A26
  + A27
  + A28
  + A29
  + A30
  + A31
  + A32
  + A33
  + A34
  + A35
  + A36
  + A37
  + A38
  + A39
  + A40
  + A41
  + A42
  + A43
  + A44
  + A45
  + A46
  + A47
  + A48
  + A49
  + A50
  + A51
  + A52
  + A53
  + A54
  + A55
  + A56
  + A57
  + A58
  + A59
  + A60
  + A61
  + A62
  + A63
  + geometry

Removed attributes:
  - id
  - CaCO3


# Split of SOC samples (Dataset-I)

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from pyproj import CRS
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


# ============================================================
# USER SETTINGS
# ============================================================

input_file = (
    "/content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/LUCAS_samples_sentinel.geojson"
)

output_folder = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/1_Feature_Selection/"
    "Split_features"
)

# Percentage reserved for independent validation
validation_size = 0.10

# Reproducible random split
random_seed = 42

# Required output CRS
output_crs = "EPSG:3763"

# Assign this CRS only when the input file has no CRS metadata
# and the coordinates are known to be WGS 84 longitude/latitude.
source_crs_if_missing = "EPSG:4326"

# Min-Max normalization range
normalization_range = (0.0, 1.0)

# False is recommended because it preserves validation values
# outside the development-data range as values below 0 or above 1.
clip_validation_values = False

# If True, the code stops when any removal column is missing.
# If False, missing columns are reported and ignored.
strict_column_removal = False


# ============================================================
# COLUMNS TO REMOVE
# ============================================================

columns_to_remove = [
    "id",
    "CaCO3",
    "CaCO3_20_",
    "Depth",
    "EC",
    "Elev",
    "K",
    "LC",
    "OC__20_30_",

    "LU",
    "CaCO3__20_",
    "N",
    "NUTS_0",
    "NUTS_1",
    "NUTS_2",
    "NUTS_3",
    "OC_20_30_",
    "Ox_Al",
    "Ox_Fe",
    "P",
    "POINTID",
    "POINTID_1",
    "TH_LAT",
    "TH_LONG",
    "pH_CaCl2",
    "pH_H2O",
]

# These columns are retained but not normalized.
# OC is assumed to be the SOC response/target variable.
columns_not_to_normalize = [
    "OC",
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def resolve_column_names(dataframe_columns, requested_columns):
    """
    Match requested field names to actual dataframe field names
    using case-insensitive matching.
    """
    lookup = {}

    for column in dataframe_columns:
        key = str(column).strip().casefold()

        if key in lookup:
            raise ValueError(
                "Duplicate field names were detected when compared "
                "without case sensitivity:\n"
                f"  - {lookup[key]}\n"
                f"  - {column}"
            )

        lookup[key] = column

    matched = []
    missing = []

    for requested_column in requested_columns:
        key = str(requested_column).strip().casefold()

        if key in lookup:
            matched.append(lookup[key])
        else:
            missing.append(requested_column)

    return matched, missing


def delete_existing_file(file_path):
    """Delete an existing output file before rewriting it."""
    if os.path.exists(file_path):
        os.remove(file_path)


# ============================================================
# VALIDATE SETTINGS
# ============================================================

if not 0 < validation_size < 1:
    raise ValueError(
        "'validation_size' must be greater than 0 and less than 1."
    )

if (
    len(normalization_range) != 2
    or normalization_range[0] >= normalization_range[1]
):
    raise ValueError(
        "'normalization_range' must contain a valid minimum "
        "and maximum value."
    )

if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"Input file was not found:\n{input_file}"
    )


# ============================================================
# READ INPUT DATA
# ============================================================

gdf = gpd.read_file(input_file)

if gdf.empty:
    raise ValueError(
        "The input spatial dataset contains no features."
    )

geometry_column = gdf.geometry.name
original_field_count = len(gdf.columns)

print("=" * 75)
print("INPUT DATA")
print("=" * 75)
print(f"Input file:               {input_file}")
print(f"Number of points:         {len(gdf):,}")
print(f"Number of fields:         {original_field_count:,}")
print(f"Original CRS:             {gdf.crs}")
print(
    "Geometry types:          "
    f"{gdf.geometry.geom_type.dropna().unique().tolist()}"
)


# ============================================================
# CHECK OR ASSIGN SOURCE CRS
# ============================================================

if gdf.crs is None:
    if source_crs_if_missing is None:
        raise ValueError(
            "The input dataset has no CRS information. Set "
            "'source_crs_if_missing' to the actual source CRS."
        )

    print(
        "\nWarning: the input file has no CRS metadata. "
        f"Assigning {source_crs_if_missing}."
    )

    # This assigns CRS metadata without transforming coordinates.
    gdf = gdf.set_crs(source_crs_if_missing)


# ============================================================
# GEOMETRY QUALITY CHECKS
# ============================================================

missing_geometry_count = int(gdf.geometry.isna().sum())

empty_geometry_count = int(
    gdf.geometry.is_empty.fillna(False).sum()
)

if missing_geometry_count > 0:
    raise ValueError(
        f"The dataset contains {missing_geometry_count:,} "
        "rows with missing geometry."
    )

if empty_geometry_count > 0:
    raise ValueError(
        f"The dataset contains {empty_geometry_count:,} "
        "rows with empty geometry."
    )

allowed_geometry_types = ["Point", "MultiPoint"]

non_point_mask = ~gdf.geometry.geom_type.isin(
    allowed_geometry_types
)

if non_point_mask.any():
    non_point_types = (
        gdf.loc[non_point_mask, geometry_column]
        .geom_type
        .unique()
        .tolist()
    )

    raise ValueError(
        "The workflow requires point geometries, but the following "
        f"geometry types were found:\n{non_point_types}"
    )


# ============================================================
# REPROJECT TO EPSG:3763
# ============================================================

required_crs = CRS.from_user_input(output_crs)
current_crs = CRS.from_user_input(gdf.crs)

if not current_crs.equals(required_crs):
    print(
        f"\nReprojecting from {gdf.crs} to {output_crs}..."
    )

    gdf = gdf.to_crs(required_crs)

if gdf.crs is None or gdf.crs.to_epsg() != 3763:
    raise RuntimeError(
        "Reprojection failed.\n"
        f"Current CRS: {gdf.crs}\n"
        f"Required CRS: {output_crs}"
    )

print(f"Output CRS:               {gdf.crs}")


# ============================================================
# REMOVE UNWANTED COLUMNS
# ============================================================

actual_columns_to_remove, missing_removal_columns = (
    resolve_column_names(
        dataframe_columns=gdf.columns,
        requested_columns=columns_to_remove,
    )
)

if geometry_column in actual_columns_to_remove:
    raise ValueError(
        f"The geometry column '{geometry_column}' cannot be removed."
    )

if missing_removal_columns:
    missing_text = "\n".join(
        f"  - {column}"
        for column in missing_removal_columns
    )

    if strict_column_removal:
        raise KeyError(
            "The following requested fields were not found:\n"
            f"{missing_text}"
        )

    print(
        "\nWarning: the following requested fields were not found "
        "and will be ignored:"
    )
    print(missing_text)

gdf = gdf.drop(
    columns=actual_columns_to_remove,
    errors="raise",
).copy()

print("\n" + "=" * 75)
print("COLUMN REMOVAL")
print("=" * 75)
print(
    f"Requested fields:         {len(columns_to_remove):,}"
)
print(
    f"Fields removed:           {len(actual_columns_to_remove):,}"
)
print(
    f"Fields remaining:         {len(gdf.columns):,}"
)

print("\nRemoved fields:")

for column in actual_columns_to_remove:
    print(f"  - {column}")


# ============================================================
# IDENTIFY COLUMNS THAT MUST NOT BE NORMALIZED
# ============================================================

actual_unscaled_columns, missing_unscaled_columns = (
    resolve_column_names(
        dataframe_columns=gdf.columns,
        requested_columns=columns_not_to_normalize,
    )
)

if missing_unscaled_columns:
    print(
        "\nWarning: these requested unscaled fields were not found:"
    )

    for column in missing_unscaled_columns:
        print(f"  - {column}")

if actual_unscaled_columns:
    print("\nFields retained without normalization:")

    for column in actual_unscaled_columns:
        print(f"  - {column}")


# ============================================================
# CREATE TEMPORARY UNIQUE ROW IDENTIFIER
# ============================================================

temporary_id = "_split_row_id"

while temporary_id in gdf.columns:
    temporary_id = "_" + temporary_id

gdf[temporary_id] = np.arange(
    len(gdf),
    dtype=np.int64,
)


# ============================================================
# RANDOM 90% / 10% SPLIT
# ============================================================

all_row_indices = np.arange(len(gdf))

development_indices, validation_indices = train_test_split(
    all_row_indices,
    test_size=validation_size,
    random_state=random_seed,
    shuffle=True,
)

gdf_development = gdf.iloc[
    development_indices
].copy()

gdf_validation = gdf.iloc[
    validation_indices
].copy()


# ============================================================
# IDENTIFY NUMERIC PREDICTOR COLUMNS
# ============================================================

numeric_columns = (
    gdf.select_dtypes(include=[np.number])
    .columns
    .tolist()
)

excluded_from_normalization = set(
    actual_unscaled_columns + [temporary_id]
)

normalization_columns = [
    column
    for column in numeric_columns
    if column not in excluded_from_normalization
]

if not normalization_columns:
    raise ValueError(
        "No numeric predictor columns remain for normalization."
    )

non_numeric_columns = [
    column
    for column in gdf.columns
    if (
        column not in numeric_columns
        and column != geometry_column
    )
]

print("\n" + "=" * 75)
print("MIN-MAX NORMALIZATION")
print("=" * 75)
print(
    f"Numeric fields to normalize: "
    f"{len(normalization_columns):,}"
)

for column in normalization_columns:
    print(f"  - {column}")

if non_numeric_columns:
    print(
        "\nNon-numeric fields retained without normalization:"
    )

    for column in non_numeric_columns:
        print(f"  - {column}")


# ============================================================
# CHECK FOR INVALID NUMERIC VALUES
# ============================================================

invalid_numeric_columns = {}

for column in normalization_columns:
    numeric_values = pd.to_numeric(
        gdf[column],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64,
        na_value=np.nan,
    )

    missing_count = int(
        np.isnan(numeric_values).sum()
    )

    infinite_count = int(
        np.isinf(numeric_values).sum()
    )

    if missing_count > 0 or infinite_count > 0:
        invalid_numeric_columns[column] = {
            "missing": missing_count,
            "infinite": infinite_count,
        }

if invalid_numeric_columns:
    error_lines = []

    for column, counts in invalid_numeric_columns.items():
        error_lines.append(
            f"  - {column}: "
            f"{counts['missing']:,} missing/non-numeric and "
            f"{counts['infinite']:,} infinite values"
        )

    raise ValueError(
        "Normalization cannot continue because the following "
        "predictor fields contain invalid values:\n"
        + "\n".join(error_lines)
    )


# ============================================================
# FIT SCALER ON DEVELOPMENT DATA ONLY
# ============================================================

gdf_development[normalization_columns] = (
    gdf_development[normalization_columns]
    .astype(np.float64)
)

gdf_validation[normalization_columns] = (
    gdf_validation[normalization_columns]
    .astype(np.float64)
)

scaler = MinMaxScaler(
    feature_range=normalization_range,
    clip=clip_validation_values,
)

# Fit using only the 90% development dataset
gdf_development.loc[
    :,
    normalization_columns,
] = scaler.fit_transform(
    gdf_development[normalization_columns]
)

# Apply the development-data scaling parameters to validation
gdf_validation.loc[
    :,
    normalization_columns,
] = scaler.transform(
    gdf_validation[normalization_columns]
)


# ============================================================
# REPORT CONSTANT PREDICTORS
# ============================================================

constant_columns = [
    column
    for column, data_range in zip(
        normalization_columns,
        scaler.data_range_,
    )
    if np.isclose(data_range, 0.0)
]

if constant_columns:
    print(
        "\nWarning: these predictors were constant in the "
        "development dataset and were normalized to 0:"
    )

    for column in constant_columns:
        print(f"  - {column}")


# ============================================================
# VERIFY NORMALIZATION
# ============================================================

minimum_range, maximum_range = normalization_range
tolerance = 1e-10

development_values = gdf_development[
    normalization_columns
].to_numpy(dtype=np.float64)

development_below_range = int(
    (
        development_values
        < minimum_range - tolerance
    ).sum()
)

development_above_range = int(
    (
        development_values
        > maximum_range + tolerance
    ).sum()
)

if development_below_range > 0 or development_above_range > 0:
    raise RuntimeError(
        "Normalization verification failed for the "
        "development dataset."
    )

validation_values = gdf_validation[
    normalization_columns
].to_numpy(dtype=np.float64)

validation_below_range = int(
    (
        validation_values
        < minimum_range - tolerance
    ).sum()
)

validation_above_range = int(
    (
        validation_values
        > maximum_range + tolerance
    ).sum()
)

if (
    not clip_validation_values
    and (
        validation_below_range > 0
        or validation_above_range > 0
    )
):
    print(
        "\nNote: some independent-validation values are outside "
        "the range [0, 1] because they are outside the minimum or "
        "maximum observed in the development dataset."
    )
    print(
        f"Values below {minimum_range}: "
        f"{validation_below_range:,}"
    )
    print(
        f"Values above {maximum_range}: "
        f"{validation_above_range:,}"
    )


# ============================================================
# RANDOMIZE ROW ORDER WITHIN EACH DATASET
# ============================================================

gdf_development = (
    gdf_development
    .sample(
        frac=1,
        random_state=random_seed,
    )
    .reset_index(drop=True)
)

gdf_validation = (
    gdf_validation
    .sample(
        frac=1,
        random_state=random_seed,
    )
    .reset_index(drop=True)
)


# ============================================================
# CHECK SPLIT COMPLETENESS AND OVERLAP
# ============================================================

development_ids = set(
    gdf_development[temporary_id].tolist()
)

validation_ids = set(
    gdf_validation[temporary_id].tolist()
)

overlapping_ids = development_ids.intersection(
    validation_ids
)

combined_ids = development_ids.union(
    validation_ids
)

if overlapping_ids:
    raise RuntimeError(
        f"Split error: {len(overlapping_ids):,} rows occur "
        "in both output datasets."
    )

if len(combined_ids) != len(gdf):
    raise RuntimeError(
        "Split error: some input rows are missing from "
        "the output datasets."
    )

if (
    len(gdf_development)
    + len(gdf_validation)
    != len(gdf)
):
    raise RuntimeError(
        "Split error: the output row total does not equal "
        "the input row total."
    )


# ============================================================
# REMOVE TEMPORARY IDENTIFIER
# ============================================================

gdf_development = gdf_development.drop(
    columns=[temporary_id]
)

gdf_validation = gdf_validation.drop(
    columns=[temporary_id]
)

gdf_development = gpd.GeoDataFrame(
    gdf_development,
    geometry=geometry_column,
    crs=output_crs,
)

gdf_validation = gpd.GeoDataFrame(
    gdf_validation,
    geometry=geometry_column,
    crs=output_crs,
)


# ============================================================
# PREPARE OUTPUT PATHS
# ============================================================

Path(output_folder).mkdir(
    parents=True,
    exist_ok=True,
)

base_name = Path(input_file).stem

development_path = os.path.join(
    output_folder,
    f"{base_name}_train_test.gpkg",
)

validation_path = os.path.join(
    output_folder,
    f"{base_name}_validation.gpkg",
)

delete_existing_file(development_path)
delete_existing_file(validation_path)


# ============================================================
# EXPORT ONLY THE TWO GEOPACKAGE FILES
# ============================================================

gdf_development.to_file(
    development_path,
    layer="development_90_percent",
    driver="GPKG",
    index=False,
)

gdf_validation.to_file(
    validation_path,
    layer="independent_validation_10_percent",
    driver="GPKG",
    index=False,
)


# ============================================================
# READ OUTPUTS BACK FOR FINAL VERIFICATION
# ============================================================

check_development = gpd.read_file(
    development_path,
    layer="development_90_percent",
)

check_validation = gpd.read_file(
    validation_path,
    layer="independent_validation_10_percent",
)

if (
    check_development.crs is None
    or check_development.crs.to_epsg() != 3763
):
    raise RuntimeError(
        "The development GeoPackage was not exported "
        "in EPSG:3763."
    )

if (
    check_validation.crs is None
    or check_validation.crs.to_epsg() != 3763
):
    raise RuntimeError(
        "The validation GeoPackage was not exported "
        "in EPSG:3763."
    )

if len(check_development) != len(gdf_development):
    raise RuntimeError(
        "The exported development row count is incorrect."
    )

if len(check_validation) != len(gdf_validation):
    raise RuntimeError(
        "The exported validation row count is incorrect."
    )

# Confirm that removed columns are absent
development_field_lookup = {
    str(column).casefold()
    for column in check_development.columns
}

validation_field_lookup = {
    str(column).casefold()
    for column in check_validation.columns
}

unexpected_fields = [
    column
    for column in columns_to_remove
    if (
        column.casefold() in development_field_lookup
        or column.casefold() in validation_field_lookup
    )
]

if unexpected_fields:
    raise RuntimeError(
        "Some requested removal fields remain in the outputs:\n"
        + "\n".join(
            f"  - {column}"
            for column in unexpected_fields
        )
    )


# ============================================================
# FINAL REPORT
# ============================================================

development_percentage = (
    len(gdf_development) / len(gdf) * 100
)

validation_percentage = (
    len(gdf_validation) / len(gdf) * 100
)

print("\n" + "=" * 75)
print("PROCESSING COMPLETED SUCCESSFULLY")
print("=" * 75)
print(f"Random seed:                    {random_seed}")
print(f"Output CRS:                     {gdf.crs}")
print(
    "Normalization fitted using:    "
    "90% development dataset only"
)
print(
    f"Normalized predictor fields:    "
    f"{len(normalization_columns):,}"
)
print(
    f"Removed fields:                 "
    f"{len(actual_columns_to_remove):,}"
)
print(
    f"Constant predictor fields:      "
    f"{len(constant_columns):,}"
)
print()
print(
    f"Original number of points:      "
    f"{len(gdf):,}"
)
print(
    f"Development points:             "
    f"{len(gdf_development):,} "
    f"({development_percentage:.2f}%)"
)
print(
    f"Independent validation points:  "
    f"{len(gdf_validation):,} "
    f"({validation_percentage:.2f}%)"
)
print(
    f"Overlapping input rows:         "
    f"{len(overlapping_ids):,}"
)
print()
print("Development GeoPackage:")
print(development_path)
print()
print("Independent validation GeoPackage:")
print(validation_path)

INPUT DATA
Input file:               /content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/LUCAS_samples_sentinel.geojson
Number of points:         428
Number of fields:         69
Original CRS:             EPSG:4326
Geometry types:          ['Point']

Reprojecting from EPSG:4326 to EPSG:3763...
Output CRS:               EPSG:3763

  - CaCO3_20_
  - OC_20_30_

COLUMN REMOVAL
Requested fields:         26
Fields removed:           24
Fields remaining:         45

Removed fields:
  - id
  - CaCO3
  - Depth
  - EC
  - Elev
  - K
  - LC
  - OC__20_30_
  - LU
  - CaCO3__20_
  - N
  - NUTS_0
  - NUTS_1
  - NUTS_2
  - NUTS_3
  - Ox_Al
  - Ox_Fe
  - P
  - POINTID
  - POINTID_1
  - TH_LAT
  - TH_LONG
  - pH_CaCl2
  - pH_H2O

Fields retained without normalization:
  - OC

MIN-MAX NORMALIZATION
Numeric fields to normalize: 40
  - LST_2018_C_mean
  - NDVI_2018_mean
  - S1_201801_VH
  - S1_201801_VV
  - S1_201802_VH
  - S1_201802_VV
  - S1_201803_VH
  - S1_201803_VV
  - S1_2

# Image mosacing

Read the image

In [2]:
from pathlib import Path

import rasterio


# ============================================================
# USER SETTINGS
# ============================================================

INPUT_FOLDER = Path(
    "/content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/GSE_2018_Imagery_raw"
)


# ============================================================
# SELECT ONLY ONE RASTER
# ============================================================

raster_files = sorted(
    list(INPUT_FOLDER.glob("*.tif"))
    + list(INPUT_FOLDER.glob("*.tiff"))
)

if not raster_files:
    raise FileNotFoundError(
        f"No TIFF files were found in:\n{INPUT_FOLDER}"
    )

# Select the first raster alphabetically
raster_path = raster_files[0]

print(f"Selected raster: {raster_path.name}")


# ============================================================
# PRINT BAND NAMES
# ============================================================

with rasterio.open(raster_path) as src:

    print(f"CRS: {src.crs}")
    print(f"Raster size: {src.width} × {src.height}")
    print(f"Number of bands: {src.count}")

    print("\nBand names:")

    for band_index in range(1, src.count + 1):

        description = src.descriptions[band_index - 1]
        tags = src.tags(band_index)

        band_name = (
            description
            or tags.get("name")
            or tags.get("band_name")
            or tags.get("long_name")
            or tags.get("DESCRIPTION")
            or f"Band_{band_index}"
        )

        print(
            f"Band {band_index}: {band_name} "
            f"| dtype={src.dtypes[band_index - 1]}"
        )

Selected raster: GSE_selected_bands_2018_PT-0000000000-0000000000.tif
CRS: EPSG:3763
Raster size: 3328 × 3328
Number of bands: 51

Band names:
Band 1: A01 | dtype=float64
Band 2: A03 | dtype=float64
Band 3: A05 | dtype=float64
Band 4: A06 | dtype=float64
Band 5: A07 | dtype=float64
Band 6: A08 | dtype=float64
Band 7: A09 | dtype=float64
Band 8: A10 | dtype=float64
Band 9: A11 | dtype=float64
Band 10: A13 | dtype=float64
Band 11: A14 | dtype=float64
Band 12: A16 | dtype=float64
Band 13: A17 | dtype=float64
Band 14: A18 | dtype=float64
Band 15: A19 | dtype=float64
Band 16: A20 | dtype=float64
Band 17: A21 | dtype=float64
Band 18: A23 | dtype=float64
Band 19: A24 | dtype=float64
Band 20: A26 | dtype=float64
Band 21: A27 | dtype=float64
Band 22: A30 | dtype=float64
Band 23: A31 | dtype=float64
Band 24: A32 | dtype=float64
Band 25: A33 | dtype=float64
Band 26: A34 | dtype=float64
Band 27: A35 | dtype=float64
Band 28: A36 | dtype=float64
Band 29: A37 | dtype=float64
Band 30: A38 | dtype=floa

In [1]:
# Optional Colab installation (run once if required):
# !pip install -q GDAL

"""
Select 33 GSE bands from every input tile and create a block-based mosaic.

Main improvements
-----------------
1. Reads the requested bands by their stored band names.
2. Verifies band order, CRS, resolution, and raster readability.
3. Uses explicit xRes/yRes with targetAlignedPixels=True, preventing:
       "-tap option cannot be used without using -tr"
4. Writes the mosaic as separate GeoTIFF blocks directly to Google Drive.
5. Creates each block locally first, then copies and verifies it in Drive.
6. Preserves completed blocks and resumes after a Colab interruption.
7. Creates a final VRT that displays all blocks as one continuous mosaic.
8. Saves CSV reports for block status and invalid source tiles.

No single very large GeoTIFF is created.
"""

import csv
import glob
import json
import math
import os
import shutil
import time
from pathlib import Path

import numpy as np
from osgeo import gdal


# ============================================================
# 1. USER SETTINGS
# ============================================================

INPUT_DIR = Path(
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/SOC_GEE_exports/"
    "GSE_2018_Imagery_raw"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/SOC_GEE_exports/"
    "GSE_2018_Imagery_mosaic"
)

OUTPUT_BLOCK_DIR = OUTPUT_DIR / "selected_band_mosaic_blocks"

# Temporary local Colab folder. It is intentionally outside Google Drive.
LOCAL_TEMP_DIR = Path("/content/GSE_2018_selected_band_mosaic_temp")

# Temporary source VRT used only while the script is running.
SELECTED_SOURCE_VRT = (
    LOCAL_TEMP_DIR / "GSE_2018_selected_bands_source_mosaic.vrt"
)

# Permanent VRT saved in Google Drive. Open this in QGIS to view all blocks
# as one continuous mosaic.
FINAL_BLOCK_MOSAIC_VRT = (
    OUTPUT_DIR / "GSE_2018_selected_33_bands_block_mosaic.vrt"
)

BLOCK_MANIFEST_CSV = (
    OUTPUT_DIR / "GSE_2018_selected_33_bands_block_manifest.csv"
)

CHECKPOINT_JSON = (
    OUTPUT_DIR / "GSE_2018_selected_33_bands_checkpoint.json"
)

INVALID_INPUT_CSV = (
    OUTPUT_DIR / "GSE_2018_invalid_input_tiles.csv"
)

INPUT_PATTERNS = [
    "GSE_selected_bands_2018_PT-*.tif",
    "GSE_selected_bands_2018_PT-*.tiff",
]

SELECTED_BAND_NAMES = [
    "A01", "A05", "A06", "A07", "A08", "A10", "A13", "A14",
    "A16", "A19", "A21", "A23", "A26", "A27", "A30", "A31",
    "A33", "A34", "A35", "A37", "A38", "A39", "A40", "A41",
    "A45", "A48", "A49", "A50", "A51", "A53", "A54", "A56",
    "A62",
]

# 2048 × 2048 pixels is safer for Google Drive than very large blocks.
# At 10 m resolution, each block is approximately 20.48 × 20.48 km.
BLOCK_WIDTH_PIXELS = 2048
BLOCK_HEIGHT_PIXELS = 2048

OUTPUT_NODATA = np.nan

# False = keep and reuse valid blocks from an earlier run.
OVERWRITE_EXISTING_BLOCKS = False

# Overviews increase file size and processing time. The final VRT can still
# be opened without them.
BUILD_BLOCK_OVERVIEWS = False

# Check every input tile before mosaicking.
CHECK_ALL_INPUT_TILES = True

# False is scientifically safer because skipping a source tile creates a gap.
# Set True only when you deliberately accept gaps in the mosaic.
SKIP_UNREADABLE_TILES = False

# Number of attempts when opening a Google Drive raster.
OPEN_RETRY_ATTEMPTS = 4
OPEN_RETRY_DELAY_SECONDS = 3

# Use only when source rasters have no band descriptions and you are certain:
# band 1=A00, band 2=A01, band 3=A02, ...
ALLOW_A00_POSITIONAL_FALLBACK = False


# ============================================================
# 2. GDAL SETTINGS
# ============================================================

gdal.UseExceptions()

gdal.SetConfigOption("GDAL_CACHEMAX", "4096")
gdal.SetConfigOption("GDAL_NUM_THREADS", "ALL_CPUS")
gdal.SetConfigOption("BIGTIFF_OVERVIEW", "YES")
gdal.SetConfigOption("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")


# ============================================================
# 3. GENERAL HELPERS
# ============================================================

def print_header(message):
    print("\n" + "=" * 100)
    print(message)
    print("=" * 100, flush=True)


def normalize_band_name(value):
    if value is None:
        return ""
    return str(value).strip().upper()


def open_raster_with_retry(
    raster_path,
    access_mode=gdal.GA_ReadOnly,
    attempts=OPEN_RETRY_ATTEMPTS,
    delay_seconds=OPEN_RETRY_DELAY_SECONDS,
):
    """
    Open a raster with retries because files mounted from Google Drive can
    occasionally fail temporarily.
    """
    raster_path = Path(raster_path)
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            dataset = gdal.Open(str(raster_path), access_mode)

            if dataset is not None:
                return dataset

        except Exception as error:
            last_error = error

        if attempt < attempts:
            print(
                f"Retrying raster open ({attempt}/{attempts}): "
                f"{raster_path.name}",
                flush=True,
            )
            time.sleep(delay_seconds)

    if last_error is not None:
        raise RuntimeError(
            f"Could not open raster after {attempts} attempts:\n"
            f"{raster_path}\nReason: {last_error}"
        )

    raise RuntimeError(
        f"Could not open raster after {attempts} attempts:\n"
        f"{raster_path}"
    )


def find_input_files():
    files = []

    for pattern in INPUT_PATTERNS:
        files.extend(glob.glob(str(INPUT_DIR / pattern)))

    files = sorted(set(files))

    if not files:
        raise FileNotFoundError(
            "No input images were found.\n"
            f"Folder checked:\n{INPUT_DIR}\n\n"
            "Patterns checked:\n"
            + "\n".join(f"  - {pattern}" for pattern in INPUT_PATTERNS)
        )

    return [Path(path) for path in files]


def get_band_name(dataset, band_index):
    band = dataset.GetRasterBand(band_index)

    candidates = [
        band.GetDescription(),
        band.GetMetadataItem("name"),
        band.GetMetadataItem("band_name"),
        band.GetMetadataItem("long_name"),
        band.GetMetadataItem("DESCRIPTION"),
        band.GetMetadataItem("NETCDF_VARNAME"),
    ]

    for candidate in candidates:
        normalized = normalize_band_name(candidate)

        if normalized:
            return normalized

    return ""


def read_band_mapping(raster_path):
    """
    Return:
      mapping: {'A01': 1, 'A05': 3, ...}
      raster_count
    """
    dataset = open_raster_with_retry(raster_path)

    mapping = {}

    for band_index in range(1, dataset.RasterCount + 1):
        band_name = get_band_name(dataset, band_index)

        if not band_name:
            continue

        if band_name in mapping:
            dataset = None
            raise ValueError(
                f"Duplicate band name '{band_name}' in:\n{raster_path}"
            )

        mapping[band_name] = band_index

    raster_count = dataset.RasterCount
    dataset = None

    if not mapping and ALLOW_A00_POSITIONAL_FALLBACK:
        mapping = {
            f"A{position:02d}": position + 1
            for position in range(raster_count)
        }

    return mapping, raster_count


def read_raster_grid_information(raster_path):
    dataset = open_raster_with_retry(raster_path)

    transform = dataset.GetGeoTransform()

    information = {
        "projection": dataset.GetProjection(),
        "x_resolution": abs(float(transform[1])),
        "y_resolution": abs(float(transform[5])),
        "rotation_x": float(transform[2]),
        "rotation_y": float(transform[4]),
        "width": dataset.RasterXSize,
        "height": dataset.RasterYSize,
        "band_count": dataset.RasterCount,
    }

    dataset = None
    return information


def raster_is_valid(
    raster_path,
    expected_band_count=None,
    attempts=3,
):
    raster_path = Path(raster_path)

    if not raster_path.is_file():
        return False

    if raster_path.stat().st_size == 0:
        return False

    try:
        dataset = open_raster_with_retry(
            raster_path,
            attempts=attempts,
            delay_seconds=2,
        )

        valid = (
            dataset.RasterXSize > 0
            and dataset.RasterYSize > 0
            and dataset.RasterCount > 0
        )

        if expected_band_count is not None:
            valid = valid and (
                dataset.RasterCount == expected_band_count
            )

        dataset = None
        return valid

    except Exception:
        return False



def expected_block_geotransform(
    source_transform,
    x_offset,
    y_offset,
):
    """Calculate the geotransform expected for one output block."""

    origin_x = (
        source_transform[0]
        + x_offset * source_transform[1]
        + y_offset * source_transform[2]
    )

    origin_y = (
        source_transform[3]
        + x_offset * source_transform[4]
        + y_offset * source_transform[5]
    )

    return (
        origin_x,
        source_transform[1],
        source_transform[2],
        origin_y,
        source_transform[4],
        source_transform[5],
    )


def output_block_is_valid(
    raster_path,
    expected_width,
    expected_height,
    expected_transform,
    expected_projection,
):
    """
    Validate a completed block before reusing it.

    The test checks:
    - file existence and non-zero size;
    - width and height;
    - 33 output bands;
    - projection;
    - geotransform;
    - stored output band names.
    """

    raster_path = Path(raster_path)

    if not raster_path.is_file():
        return False

    if raster_path.stat().st_size == 0:
        return False

    try:
        dataset = open_raster_with_retry(
            raster_path,
            attempts=3,
            delay_seconds=2,
        )

        valid = (
            dataset.RasterXSize == expected_width
            and dataset.RasterYSize == expected_height
            and dataset.RasterCount == len(SELECTED_BAND_NAMES)
            and dataset.GetProjection() == expected_projection
            and np.allclose(
                dataset.GetGeoTransform(),
                expected_transform,
                rtol=0,
                atol=1e-7,
            )
        )

        if valid:
            for band_index, expected_name in enumerate(
                SELECTED_BAND_NAMES,
                start=1,
            ):
                actual_name = get_band_name(
                    dataset,
                    band_index,
                )

                if normalize_band_name(actual_name) != expected_name:
                    valid = False
                    break

        dataset = None
        return valid

    except Exception:
        return False


def save_checkpoint(
    total_blocks,
    completed_blocks,
    remaining_blocks,
    last_completed_block=None,
    current_status="running",
):
    """Save an atomic progress checkpoint in Google Drive."""

    checkpoint = {
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "status": current_status,
        "total_blocks": int(total_blocks),
        "completed_blocks": int(completed_blocks),
        "remaining_blocks": int(remaining_blocks),
        "last_completed_block": last_completed_block,
        "overwrite_existing_blocks": bool(
            OVERWRITE_EXISTING_BLOCKS
        ),
        "selected_band_count": len(SELECTED_BAND_NAMES),
        "block_width_pixels": BLOCK_WIDTH_PIXELS,
        "block_height_pixels": BLOCK_HEIGHT_PIXELS,
    }

    temporary_path = CHECKPOINT_JSON.with_suffix(
        ".json.tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file_object:
        json.dump(
            checkpoint,
            file_object,
            indent=2,
        )

    os.replace(
        temporary_path,
        CHECKPOINT_JSON,
    )


def set_output_band_names(raster_path, band_names):
    dataset = open_raster_with_retry(
        raster_path,
        access_mode=gdal.GA_Update,
    )

    if dataset.RasterCount != len(band_names):
        actual_count = dataset.RasterCount
        dataset = None

        raise RuntimeError(
            f"Unexpected band count in output raster:\n"
            f"{raster_path}\n"
            f"Expected: {len(band_names)}\n"
            f"Found: {actual_count}"
        )

    for band_index, band_name in enumerate(band_names, start=1):
        band = dataset.GetRasterBand(band_index)
        band.SetDescription(band_name)
        band.SetMetadataItem("name", band_name)

    dataset.FlushCache()
    dataset = None


def save_csv(rows, output_csv, fieldnames):
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    with open(
        output_csv,
        "w",
        newline="",
        encoding="utf-8",
    ) as file_object:
        writer = csv.DictWriter(
            file_object,
            fieldnames=fieldnames,
        )
        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# 4. INPUT VALIDATION
# ============================================================

def validate_input_tiles(
    input_files,
    selected_band_indices,
    reference_information,
):
    """
    Verify readability, selected bands, band order, CRS, resolution, and
    absence of raster rotation.
    """
    valid_files = []
    invalid_rows = []

    print("\nChecking selected bands and raster properties in all tiles...")

    for file_number, raster_path in enumerate(input_files, start=1):
        try:
            mapping, _ = read_band_mapping(raster_path)
            information = read_raster_grid_information(raster_path)

            missing_bands = [
                band_name
                for band_name in SELECTED_BAND_NAMES
                if band_name not in mapping
            ]

            if missing_bands:
                raise ValueError(
                    "Missing selected bands: "
                    + ", ".join(missing_bands)
                )

            current_indices = [
                mapping[band_name]
                for band_name in SELECTED_BAND_NAMES
            ]

            if current_indices != selected_band_indices:
                raise ValueError(
                    "Selected-band order differs from the reference tile."
                )

            if information["projection"] != reference_information["projection"]:
                raise ValueError("Projection differs from the reference tile.")

            if not np.isclose(
                information["x_resolution"],
                reference_information["x_resolution"],
                rtol=0,
                atol=1e-9,
            ):
                raise ValueError(
                    "X resolution differs from the reference tile."
                )

            if not np.isclose(
                information["y_resolution"],
                reference_information["y_resolution"],
                rtol=0,
                atol=1e-9,
            ):
                raise ValueError(
                    "Y resolution differs from the reference tile."
                )

            if (
                not np.isclose(information["rotation_x"], 0.0)
                or not np.isclose(information["rotation_y"], 0.0)
            ):
                raise ValueError(
                    "Rotated rasters are not supported by this workflow."
                )

            valid_files.append(raster_path)

        except Exception as error:
            invalid_rows.append({
                "file": str(raster_path),
                "error": str(error),
            })

            print(
                "\nInput tile problem:\n"
                f"  File: {raster_path}\n"
                f"  Error: {error}",
                flush=True,
            )

            if not SKIP_UNREADABLE_TILES:
                save_csv(
                    invalid_rows,
                    INVALID_INPUT_CSV,
                    ["file", "error"],
                )

                raise RuntimeError(
                    "Input validation stopped because at least one tile "
                    "is unreadable or inconsistent.\n"
                    f"Report saved to:\n{INVALID_INPUT_CSV}\n\n"
                    "Set SKIP_UNREADABLE_TILES=True only when you accept "
                    "a spatial gap in the final mosaic."
                ) from error

        if (
            file_number % 25 == 0
            or file_number == len(input_files)
        ):
            print(
                f"Checked {file_number}/{len(input_files)} tiles.",
                flush=True,
            )

    if invalid_rows:
        save_csv(
            invalid_rows,
            INVALID_INPUT_CSV,
            ["file", "error"],
        )

    if not valid_files:
        raise RuntimeError("No valid input rasters remain.")

    print(f"\nValid input tiles: {len(valid_files)}")
    print(f"Invalid input tiles: {len(invalid_rows)}")

    return valid_files


# ============================================================
# 5. BUILD SELECTED-BAND SOURCE VRT
# ============================================================

def create_selected_source_vrt(
    input_files,
    selected_band_indices,
    x_resolution,
    y_resolution,
):
    """
    Build a VRT containing only the requested 33 bands.

    Important:
    targetAlignedPixels=True requires explicit xRes and yRes.
    """
    if SELECTED_SOURCE_VRT.exists():
        SELECTED_SOURCE_VRT.unlink()

    print("\nBuilding selected-band source VRT...")

    vrt_arguments = {
        "bandList": selected_band_indices,
        "resolution": "user",
        "xRes": x_resolution,
        "yRes": y_resolution,
        "targetAlignedPixels": True,
        "resampleAlg": "nearest",
        "VRTNodata": "nan",
    }

    try:
        vrt_options = gdal.BuildVRTOptions(
            **vrt_arguments,
            ignoreSrcMaskBand=True,
        )
    except TypeError:
        vrt_options = gdal.BuildVRTOptions(**vrt_arguments)

    vrt_dataset = gdal.BuildVRT(
        str(SELECTED_SOURCE_VRT),
        [str(path) for path in input_files],
        options=vrt_options,
    )

    if vrt_dataset is None:
        raise RuntimeError(
            "GDAL could not create the selected-band source VRT."
        )

    if vrt_dataset.RasterCount != len(SELECTED_BAND_NAMES):
        actual_count = vrt_dataset.RasterCount
        vrt_dataset = None

        raise RuntimeError(
            f"Selected-band VRT has {actual_count} bands; "
            f"{len(SELECTED_BAND_NAMES)} were expected."
        )

    for band_index, band_name in enumerate(
        SELECTED_BAND_NAMES,
        start=1,
    ):
        band = vrt_dataset.GetRasterBand(band_index)
        band.SetDescription(band_name)
        band.SetMetadataItem("name", band_name)

    vrt_dataset.FlushCache()
    vrt_dataset = None

    print(f"Selected-band VRT created:\n{SELECTED_SOURCE_VRT}")


# ============================================================
# 6. WRITE BLOCKS
# ============================================================

def create_output_block(
    source_vrt,
    x_offset,
    y_offset,
    block_width,
    block_height,
    local_temp_path,
    drive_output_path,
):
    if local_temp_path.exists():
        local_temp_path.unlink()

    drive_partial_path = drive_output_path.with_suffix(
        ".partial.tif"
    )

    if drive_partial_path.exists():
        drive_partial_path.unlink()

    translate_options = gdal.TranslateOptions(
        format="GTiff",
        srcWin=[
            x_offset,
            y_offset,
            block_width,
            block_height,
        ],
        outputType=gdal.GDT_Float32,
        noData=OUTPUT_NODATA,
        creationOptions=[
            "BIGTIFF=YES",
            "TILED=YES",
            "COMPRESS=DEFLATE",
            "PREDICTOR=3",
            "ZLEVEL=6",
            "BLOCKXSIZE=512",
            "BLOCKYSIZE=512",
            "NUM_THREADS=ALL_CPUS",
        ],
    )

    block_dataset = gdal.Translate(
        str(local_temp_path),
        source_vrt,
        options=translate_options,
    )

    if block_dataset is None:
        raise RuntimeError(
            f"GDAL failed to create local block:\n{local_temp_path}"
        )

    block_dataset.FlushCache()
    block_dataset = None

    set_output_band_names(
        local_temp_path,
        SELECTED_BAND_NAMES,
    )

    if BUILD_BLOCK_OVERVIEWS:
        overview_dataset = open_raster_with_retry(
            local_temp_path,
            access_mode=gdal.GA_Update,
        )

        overview_dataset.BuildOverviews(
            "AVERAGE",
            [2, 4, 8, 16],
        )

        overview_dataset = None

    # Copy locally completed file to a temporary Drive filename.
    shutil.copy2(
        local_temp_path,
        drive_partial_path,
    )

    # Rename only after the copy is complete.
    os.replace(
        drive_partial_path,
        drive_output_path,
    )

    try:
        os.sync()
    except AttributeError:
        pass

    if not raster_is_valid(
        drive_output_path,
        expected_band_count=len(SELECTED_BAND_NAMES),
        attempts=OPEN_RETRY_ATTEMPTS,
    ):
        raise RuntimeError(
            "The copied Google Drive block failed validation:\n"
            f"{drive_output_path}"
        )

    local_temp_path.unlink(missing_ok=True)


# ============================================================
# 7. FINAL VRT FROM OUTPUT BLOCKS
# ============================================================

def build_final_block_vrt(
    block_files,
    x_resolution,
    y_resolution,
):
    if FINAL_BLOCK_MOSAIC_VRT.exists():
        FINAL_BLOCK_MOSAIC_VRT.unlink()

    print("\nBuilding final VRT from completed blocks...")

    vrt_options = gdal.BuildVRTOptions(
        resolution="user",
        xRes=x_resolution,
        yRes=y_resolution,
        targetAlignedPixels=True,
        resampleAlg="nearest",
        VRTNodata="nan",
    )

    vrt_dataset = gdal.BuildVRT(
        str(FINAL_BLOCK_MOSAIC_VRT),
        [str(path) for path in block_files],
        options=vrt_options,
    )

    if vrt_dataset is None:
        raise RuntimeError(
            "GDAL could not create the final block-mosaic VRT."
        )

    for band_index, band_name in enumerate(
        SELECTED_BAND_NAMES,
        start=1,
    ):
        band = vrt_dataset.GetRasterBand(band_index)
        band.SetDescription(band_name)
        band.SetMetadataItem("name", band_name)

    vrt_dataset.FlushCache()
    vrt_dataset = None

    print(f"Final block mosaic VRT:\n{FINAL_BLOCK_MOSAIC_VRT}")


# ============================================================
# 8. MAIN WORKFLOW
# ============================================================

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_BLOCK_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_TEMP_DIR.mkdir(parents=True, exist_ok=True)

    input_files = find_input_files()

    print_header("GSE 2018 SELECTED-BAND BLOCK MOSAIC")

    print(f"Input folder: {INPUT_DIR}")
    print(f"Input tiles: {len(input_files)}")
    print(f"Selected bands: {len(SELECTED_BAND_NAMES)}")
    print(f"Output block folder: {OUTPUT_BLOCK_DIR}")

    print("\nSelected bands:")

    for number, band_name in enumerate(
        SELECTED_BAND_NAMES,
        start=1,
    ):
        print(f"  {number:>2}. {band_name}")

    # --------------------------------------------------------
    # Reference tile and requested band indices
    # --------------------------------------------------------

    reference_tile = input_files[0]

    reference_mapping, reference_band_count = read_band_mapping(
        reference_tile
    )

    reference_information = read_raster_grid_information(
        reference_tile
    )

    print(f"\nReference tile:\n{reference_tile}")
    print(f"Reference tile band count: {reference_band_count}")
    print(
        "Reference resolution: "
        f"{reference_information['x_resolution']} × "
        f"{reference_information['y_resolution']}"
    )

    missing_reference_bands = [
        band_name
        for band_name in SELECTED_BAND_NAMES
        if band_name not in reference_mapping
    ]

    if missing_reference_bands:
        available_names = sorted(reference_mapping.keys())

        raise ValueError(
            "The reference tile does not contain all selected bands.\n\n"
            "Missing selected bands:\n"
            + "\n".join(
                f"  - {band_name}"
                for band_name in missing_reference_bands
            )
            + "\n\nDetected band names:\n"
            + (
                "\n".join(
                    f"  - {band_name}"
                    for band_name in available_names
                )
                if available_names
                else "  No stored band names were detected."
            )
            + "\n\nSet ALLOW_A00_POSITIONAL_FALLBACK=True only when "
              "band 1=A00, band 2=A01, band 3=A02, and so on."
        )

    selected_band_indices = [
        reference_mapping[band_name]
        for band_name in SELECTED_BAND_NAMES
    ]

    print("\nSelected GDAL band indices:")

    for band_name, band_index in zip(
        SELECTED_BAND_NAMES,
        selected_band_indices,
    ):
        print(f"  {band_name}: source band {band_index}")

    # --------------------------------------------------------
    # Validate all source tiles
    # --------------------------------------------------------

    if CHECK_ALL_INPUT_TILES:
        valid_input_files = validate_input_tiles(
            input_files=input_files,
            selected_band_indices=selected_band_indices,
            reference_information=reference_information,
        )
    else:
        valid_input_files = input_files

    # --------------------------------------------------------
    # Build 33-band VRT using explicit resolution
    # --------------------------------------------------------

    create_selected_source_vrt(
        input_files=valid_input_files,
        selected_band_indices=selected_band_indices,
        x_resolution=reference_information["x_resolution"],
        y_resolution=reference_information["y_resolution"],
    )

    source_vrt = open_raster_with_retry(
        SELECTED_SOURCE_VRT
    )

    mosaic_width = source_vrt.RasterXSize
    mosaic_height = source_vrt.RasterYSize
    mosaic_band_count = source_vrt.RasterCount
    mosaic_transform = source_vrt.GetGeoTransform()
    mosaic_projection = source_vrt.GetProjection()

    if mosaic_band_count != len(SELECTED_BAND_NAMES):
        source_vrt = None

        raise RuntimeError(
            f"Expected {len(SELECTED_BAND_NAMES)} bands, "
            f"but the source VRT contains {mosaic_band_count}."
        )

    number_of_block_columns = math.ceil(
        mosaic_width / BLOCK_WIDTH_PIXELS
    )

    number_of_block_rows = math.ceil(
        mosaic_height / BLOCK_HEIGHT_PIXELS
    )

    total_blocks = (
        number_of_block_columns
        * number_of_block_rows
    )

    print_header("MOSAIC GRID")

    print(f"Mosaic columns: {mosaic_width:,}")
    print(f"Mosaic rows: {mosaic_height:,}")
    print(f"Mosaic bands: {mosaic_band_count}")
    print(
        "Mosaic resolution: "
        f"{abs(mosaic_transform[1])} × "
        f"{abs(mosaic_transform[5])}"
    )
    print(
        f"Output block size: "
        f"{BLOCK_WIDTH_PIXELS} × {BLOCK_HEIGHT_PIXELS} pixels"
    )
    print(
        f"Output block grid: "
        f"{number_of_block_rows} rows × "
        f"{number_of_block_columns} columns"
    )
    print(f"Total blocks: {total_blocks:,}")

    # --------------------------------------------------------
    # Build the complete block plan and detect completed blocks
    # --------------------------------------------------------

    manifest_fieldnames = [
        "block_sequence",
        "block_id",
        "block_row",
        "block_column",
        "x_offset_pixels",
        "y_offset_pixels",
        "width_pixels",
        "height_pixels",
        "output_file",
        "status",
        "file_size_bytes",
    ]

    block_plan = []
    block_sequence = 0

    print_header("SCANNING EXISTING OUTPUT BLOCKS")

    for block_row in range(number_of_block_rows):
        y_offset = block_row * BLOCK_HEIGHT_PIXELS

        block_height = min(
            BLOCK_HEIGHT_PIXELS,
            mosaic_height - y_offset,
        )

        for block_column in range(number_of_block_columns):
            x_offset = block_column * BLOCK_WIDTH_PIXELS

            block_width = min(
                BLOCK_WIDTH_PIXELS,
                mosaic_width - x_offset,
            )

            block_sequence += 1

            block_id = (
                f"r{block_row + 1:03d}_"
                f"c{block_column + 1:03d}"
            )

            output_name = (
                "GSE_selected_33_bands_2018_PT_"
                f"{block_id}.tif"
            )

            drive_output_path = (
                OUTPUT_BLOCK_DIR / output_name
            )

            expected_transform = expected_block_geotransform(
                mosaic_transform,
                x_offset,
                y_offset,
            )

            existing_valid = (
                output_block_is_valid(
                    drive_output_path,
                    expected_width=block_width,
                    expected_height=block_height,
                    expected_transform=expected_transform,
                    expected_projection=mosaic_projection,
                )
                and not OVERWRITE_EXISTING_BLOCKS
            )

            block_plan.append({
                "block_sequence": block_sequence,
                "block_id": block_id,
                "block_row": block_row + 1,
                "block_column": block_column + 1,
                "x_offset_pixels": x_offset,
                "y_offset_pixels": y_offset,
                "width_pixels": block_width,
                "height_pixels": block_height,
                "output_file": drive_output_path,
                "expected_transform": expected_transform,
                "status": (
                    "existing"
                    if existing_valid
                    else "pending"
                ),
            })

            if (
                block_sequence % 25 == 0
                or block_sequence == total_blocks
            ):
                print(
                    f"Scanned {block_sequence}/"
                    f"{total_blocks} blocks...",
                    flush=True,
                )

    existing_count = sum(
        block["status"] == "existing"
        for block in block_plan
    )

    remaining_plan = [
        block
        for block in block_plan
        if block["status"] != "existing"
    ]

    print(f"\nValid completed blocks: {existing_count}")
    print(f"Blocks remaining: {len(remaining_plan)}")

    if remaining_plan:
        first_missing = remaining_plan[0]

        print(
            "Processing will resume from:\n"
            f"  [{first_missing['block_sequence']}/"
            f"{total_blocks}] "
            f"{first_missing['block_id']}"
        )

    else:
        print(
            "All expected output blocks already exist. "
            "No block will be recreated."
        )

    def current_manifest_rows():
        rows = []

        for block in block_plan:
            output_path = block["output_file"]

            rows.append({
                "block_sequence": block[
                    "block_sequence"
                ],
                "block_id": block["block_id"],
                "block_row": block["block_row"],
                "block_column": block[
                    "block_column"
                ],
                "x_offset_pixels": block[
                    "x_offset_pixels"
                ],
                "y_offset_pixels": block[
                    "y_offset_pixels"
                ],
                "width_pixels": block[
                    "width_pixels"
                ],
                "height_pixels": block[
                    "height_pixels"
                ],
                "output_file": str(output_path),
                "status": block["status"],
                "file_size_bytes": (
                    output_path.stat().st_size
                    if output_path.exists()
                    else 0
                ),
            })

        return rows

    save_csv(
        current_manifest_rows(),
        BLOCK_MANIFEST_CSV,
        manifest_fieldnames,
    )

    save_checkpoint(
        total_blocks=total_blocks,
        completed_blocks=existing_count,
        remaining_blocks=len(remaining_plan),
        last_completed_block=None,
        current_status=(
            "complete"
            if not remaining_plan
            else "running"
        ),
    )

    # --------------------------------------------------------
    # Process only missing or invalid blocks
    # --------------------------------------------------------

    for missing_number, block in enumerate(
        remaining_plan,
        start=1,
    ):
        drive_output_path = block["output_file"]

        local_temp_path = (
            LOCAL_TEMP_DIR
            / f"{drive_output_path.name}.partial.tif"
        )

        # Remove an invalid or interrupted output before retrying.
        drive_output_path.unlink(
            missing_ok=True
        )

        local_temp_path.unlink(
            missing_ok=True
        )

        print(
            f"\n[{block['block_sequence']}/{total_blocks}] "
            f"Creating {block['block_id']} | "
            f"remaining task {missing_number}/"
            f"{len(remaining_plan)} | "
            f"x={block['x_offset_pixels']}, "
            f"y={block['y_offset_pixels']}, "
            f"size={block['width_pixels']}×"
            f"{block['height_pixels']}",
            flush=True,
        )

        try:
            create_output_block(
                source_vrt=source_vrt,
                x_offset=block["x_offset_pixels"],
                y_offset=block["y_offset_pixels"],
                block_width=block["width_pixels"],
                block_height=block["height_pixels"],
                local_temp_path=local_temp_path,
                drive_output_path=drive_output_path,
            )

            if not output_block_is_valid(
                drive_output_path,
                expected_width=block["width_pixels"],
                expected_height=block["height_pixels"],
                expected_transform=block[
                    "expected_transform"
                ],
                expected_projection=mosaic_projection,
            ):
                raise RuntimeError(
                    "The newly created block did not "
                    "match the expected block grid."
                )

            block["status"] = "created"

            completed_count = sum(
                item["status"] in {
                    "existing",
                    "created",
                }
                for item in block_plan
            )

            remaining_count = (
                total_blocks - completed_count
            )

            print(
                "Saved and verified in Google Drive:\n"
                f"  {drive_output_path}",
                flush=True,
            )

            # Save progress immediately after every completed block.
            save_csv(
                current_manifest_rows(),
                BLOCK_MANIFEST_CSV,
                manifest_fieldnames,
            )

            save_checkpoint(
                total_blocks=total_blocks,
                completed_blocks=completed_count,
                remaining_blocks=remaining_count,
                last_completed_block=block[
                    "block_id"
                ],
                current_status=(
                    "complete"
                    if remaining_count == 0
                    else "running"
                ),
            )

        except Exception:
            block["status"] = "failed"

            save_csv(
                current_manifest_rows(),
                BLOCK_MANIFEST_CSV,
                manifest_fieldnames,
            )

            completed_count = sum(
                item["status"] in {
                    "existing",
                    "created",
                }
                for item in block_plan
            )

            save_checkpoint(
                total_blocks=total_blocks,
                completed_blocks=completed_count,
                remaining_blocks=(
                    total_blocks - completed_count
                ),
                last_completed_block=block[
                    "block_id"
                ],
                current_status="failed",
            )

            raise

    source_vrt = None

    completed_block_files = sorted(
        block["output_file"]
        for block in block_plan
    )

    valid_completed_blocks = []

    for block in block_plan:
        if output_block_is_valid(
            block["output_file"],
            expected_width=block["width_pixels"],
            expected_height=block["height_pixels"],
            expected_transform=block[
                "expected_transform"
            ],
            expected_projection=mosaic_projection,
        ):
            valid_completed_blocks.append(
                block["output_file"]
            )

    if len(valid_completed_blocks) != total_blocks:
        raise RuntimeError(
            f"Expected {total_blocks} valid completed blocks, "
            f"but found {len(valid_completed_blocks)}."
        )

    # --------------------------------------------------------
    # Build permanent whole-mosaic VRT
    # --------------------------------------------------------

    build_final_block_vrt(
        block_files=valid_completed_blocks,
        x_resolution=reference_information["x_resolution"],
        y_resolution=reference_information["y_resolution"],
    )

    print_header("PROCESSING COMPLETED")

    print(f"Completed GeoTIFF blocks: {len(valid_completed_blocks)}")
    print(f"\nBlock folder:\n{OUTPUT_BLOCK_DIR}")
    print(f"\nWhole-mosaic VRT:\n{FINAL_BLOCK_MOSAIC_VRT}")
    print(f"\nBlock manifest:\n{BLOCK_MANIFEST_CSV}")
    print(f"\nCheckpoint:\n{CHECKPOINT_JSON}")

    if INVALID_INPUT_CSV.exists():
        print(f"\nInput problem report:\n{INVALID_INPUT_CSV}")

    print(
        "\nOpen the final VRT in QGIS or another GDAL-compatible "
        "application to view the blocks as one continuous 33-band mosaic."
    )


if __name__ == "__main__":
    main()



GSE 2018 SELECTED-BAND BLOCK MOSAIC
Input folder: /content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/GSE_2018_Imagery_raw
Input tiles: 162
Selected bands: 33
Output block folder: /content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/GSE_2018_Imagery_mosaic/selected_band_mosaic_blocks

Selected bands:
   1. A01
   2. A05
   3. A06
   4. A07
   5. A08
   6. A10
   7. A13
   8. A14
   9. A16
  10. A19
  11. A21
  12. A23
  13. A26
  14. A27
  15. A30
  16. A31
  17. A33
  18. A34
  19. A35
  20. A37
  21. A38
  22. A39
  23. A40
  24. A41
  25. A45
  26. A48
  27. A49
  28. A50
  29. A51
  30. A53
  31. A54
  32. A56
  33. A62

Reference tile:
/content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/GSE_2018_Imagery_raw/GSE_selected_bands_2018_PT-0000000000-0000000000.tif
Reference tile band count: 51
Reference resolution: 10.0 × 10.0

Selected GDAL band indices:
  A01: source band 1
  A05: source band 3
  A06: so

# One Mosaic Image

In [2]:
# Optional Colab installation (run once if required):
# !pip install -q GDAL

"""
Convert the completed 33-band mainland Portugal VRT mosaic into one physical
BigTIFF GeoTIFF.

Important
---------
The VRT is already a single logical raster and is normally the safer option
for processing. This script creates one physical .tif file only when that is
specifically required.

The output is written first as:
    *.partial.tif

It is renamed to the final filename only after GDAL finishes and the raster
passes validation. A partially written GeoTIFF cannot reliably resume from
the middle; after a Colab interruption, an incomplete partial file is deleted
and the one-file conversion restarts. The original 406 source blocks remain
unchanged.
"""

import os
import time
from pathlib import Path

from osgeo import gdal


# ============================================================
# 1. USER SETTINGS
# ============================================================

INPUT_VRT = Path(
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/SOC_GEE_exports/"
    "GSE_2018_Imagery_mosaic/"
    "GSE_2018_selected_33_bands_block_mosaic.vrt"
)

OUTPUT_TIF = Path(
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/SOC_GEE_exports/"
    "GSE_2018_Imagery_mosaic/"
    "GSE_2018_selected_33_bands_mainland_Portugal.tif"
)

PARTIAL_TIF = OUTPUT_TIF.with_suffix(".partial.tif")

EXPECTED_BANDS = 33

OVERWRITE_EXISTING = False

# Building overviews can take substantial additional time and disk space.
BUILD_OVERVIEWS = False
OVERVIEW_LEVELS = [2, 4, 8, 16, 32, 64]


# ============================================================
# 2. GDAL SETTINGS
# ============================================================

gdal.UseExceptions()

gdal.SetConfigOption("GDAL_CACHEMAX", "4096")
gdal.SetConfigOption("GDAL_NUM_THREADS", "ALL_CPUS")
gdal.SetConfigOption("BIGTIFF_OVERVIEW", "YES")
gdal.SetConfigOption("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")


# ============================================================
# 3. HELPERS
# ============================================================

def header(message):
    print("\n" + "=" * 100)
    print(message)
    print("=" * 100, flush=True)


def raster_information(path):
    dataset = gdal.Open(str(path), gdal.GA_ReadOnly)

    if dataset is None:
        raise RuntimeError(f"Could not open raster:\n{path}")

    information = {
        "width": dataset.RasterXSize,
        "height": dataset.RasterYSize,
        "bands": dataset.RasterCount,
        "projection": dataset.GetProjection(),
        "geotransform": dataset.GetGeoTransform(),
        "descriptions": [
            dataset.GetRasterBand(index).GetDescription()
            for index in range(1, dataset.RasterCount + 1)
        ],
    }

    dataset = None
    return information


def output_matches_source(output_path, source_information):
    output_path = Path(output_path)

    if not output_path.exists():
        return False

    if output_path.stat().st_size == 0:
        return False

    try:
        output_information = raster_information(output_path)

        return (
            output_information["width"] == source_information["width"]
            and output_information["height"] == source_information["height"]
            and output_information["bands"] == source_information["bands"]
            and output_information["projection"] == source_information["projection"]
            and all(
                abs(a - b) <= 1e-7
                for a, b in zip(
                    output_information["geotransform"],
                    source_information["geotransform"],
                )
            )
        )

    except Exception:
        return False


def restore_band_descriptions(output_path, descriptions):
    dataset = gdal.Open(str(output_path), gdal.GA_Update)

    if dataset is None:
        raise RuntimeError(
            f"Could not reopen output for metadata update:\n{output_path}"
        )

    for band_index, description in enumerate(descriptions, start=1):
        if description:
            band = dataset.GetRasterBand(band_index)
            band.SetDescription(description)
            band.SetMetadataItem("name", description)

    dataset.FlushCache()
    dataset = None


# ============================================================
# 4. MAIN
# ============================================================

def main():
    header("CREATE ONE PHYSICAL 33-BAND MAINLAND PORTUGAL MOSAIC")

    if not INPUT_VRT.exists():
        raise FileNotFoundError(
            "Input VRT was not found:\n"
            f"{INPUT_VRT}"
        )

    OUTPUT_TIF.parent.mkdir(parents=True, exist_ok=True)

    source_information = raster_information(INPUT_VRT)

    print(f"Input VRT: {INPUT_VRT}")
    print(f"Columns: {source_information['width']:,}")
    print(f"Rows: {source_information['height']:,}")
    print(f"Bands: {source_information['bands']}")
    print(f"Output GeoTIFF: {OUTPUT_TIF}")

    if source_information["bands"] != EXPECTED_BANDS:
        raise RuntimeError(
            f"Expected {EXPECTED_BANDS} bands, but the VRT contains "
            f"{source_information['bands']}."
        )

    # A completed final output is reused unless overwrite is enabled.
    if (
        output_matches_source(OUTPUT_TIF, source_information)
        and not OVERWRITE_EXISTING
    ):
        print("\nA valid final GeoTIFF already exists. Processing skipped.")
        print(OUTPUT_TIF)
        return

    if OUTPUT_TIF.exists():
        if not OVERWRITE_EXISTING:
            raise RuntimeError(
                "An output file exists but failed validation:\n"
                f"{OUTPUT_TIF}\n\n"
                "Inspect or remove it, or set OVERWRITE_EXISTING=True."
            )

        print("\nRemoving the existing final output...")
        OUTPUT_TIF.unlink()

    # A completed partial file may remain when conversion finished but Colab
    # stopped before the atomic rename.
    if PARTIAL_TIF.exists():
        if output_matches_source(PARTIAL_TIF, source_information):
            print(
                "\nA complete and valid partial GeoTIFF was found. "
                "Renaming it to the final output..."
            )
            os.replace(PARTIAL_TIF, OUTPUT_TIF)
            print(f"Final GeoTIFF:\n{OUTPUT_TIF}")
            return

        print("\nRemoving an incomplete or invalid partial GeoTIFF...")
        PARTIAL_TIF.unlink()

    print(
        "\nWriting one tiled, compressed BigTIFF directly to Google Drive."
    )
    print(
        "This may require substantial time and storage. Do not disconnect "
        "the Colab runtime during this step."
    )

    creation_options = [
        "BIGTIFF=YES",
        "TILED=YES",
        "COMPRESS=DEFLATE",
        "PREDICTOR=3",
        "ZLEVEL=6",
        "BLOCKXSIZE=512",
        "BLOCKYSIZE=512",
        "NUM_THREADS=ALL_CPUS",
        "SPARSE_OK=TRUE",
    ]

    translate_options = gdal.TranslateOptions(
        format="GTiff",
        outputType=gdal.GDT_Float32,
        noData=float("nan"),
        creationOptions=creation_options,
        callback=gdal.TermProgress_nocb,
    )

    start_time = time.time()

    source_dataset = gdal.Open(str(INPUT_VRT), gdal.GA_ReadOnly)

    if source_dataset is None:
        raise RuntimeError(f"Could not open input VRT:\n{INPUT_VRT}")

    output_dataset = gdal.Translate(
        str(PARTIAL_TIF),
        source_dataset,
        options=translate_options,
    )

    source_dataset = None

    if output_dataset is None:
        raise RuntimeError(
            "GDAL could not create the physical GeoTIFF mosaic."
        )

    output_dataset.FlushCache()
    output_dataset = None

    restore_band_descriptions(
        PARTIAL_TIF,
        source_information["descriptions"],
    )

    if BUILD_OVERVIEWS:
        print("\nBuilding raster overviews...")

        overview_dataset = gdal.Open(
            str(PARTIAL_TIF),
            gdal.GA_Update,
        )

        if overview_dataset is None:
            raise RuntimeError(
                f"Could not reopen output for overviews:\n{PARTIAL_TIF}"
            )

        overview_dataset.BuildOverviews(
            "AVERAGE",
            OVERVIEW_LEVELS,
            callback=gdal.TermProgress_nocb,
        )

        overview_dataset = None

    if not output_matches_source(PARTIAL_TIF, source_information):
        raise RuntimeError(
            "The generated partial GeoTIFF failed final validation.\n"
            f"It has been retained for inspection:\n{PARTIAL_TIF}"
        )

    os.replace(PARTIAL_TIF, OUTPUT_TIF)

    try:
        os.sync()
    except AttributeError:
        pass

    runtime_hours = (time.time() - start_time) / 3600
    output_size_gb = OUTPUT_TIF.stat().st_size / (1024 ** 3)

    header("ONE-FILE MOSAIC COMPLETED")

    print(f"Final GeoTIFF:\n{OUTPUT_TIF}")
    print(f"File size: {output_size_gb:.2f} GiB")
    print(f"Runtime: {runtime_hours:.2f} hours")
    print(f"Bands: {source_information['bands']}")
    print(
        f"Dimensions: {source_information['width']:,} × "
        f"{source_information['height']:,}"
    )


if __name__ == "__main__":
    main()



CREATE ONE PHYSICAL 33-BAND MAINLAND PORTUGAL MOSAIC
Input VRT: /content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/GSE_2018_Imagery_mosaic/GSE_2018_selected_33_bands_block_mosaic.vrt
Columns: 28,096
Rows: 57,433
Bands: 33
Output GeoTIFF: /content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/GSE_2018_Imagery_mosaic/GSE_2018_selected_33_bands_mainland_Portugal.tif

Writing one tiled, compressed BigTIFF directly to Google Drive.
This may require substantial time and storage. Do not disconnect the Colab runtime during this step.

ONE-FILE MOSAIC COMPLETED
Final GeoTIFF:
/content/drive/MyDrive/1.SOC_Estimation/1.Final_SOC_estimation/SOC_GEE_exports/GSE_2018_Imagery_mosaic/GSE_2018_selected_33_bands_mainland_Portugal.tif
File size: 78.72 GiB
Runtime: 0.64 hours
Bands: 33
Dimensions: 28,096 × 57,433
